# Finding patterns in a tumor 🔬

Forty-one women had a piece of their breast tumor photographed with a machine
that measures **36 different proteins in every single cell**.

That's about 200,000 cells. Nobody can look at that by eye. So we'll teach a
computer to find the patterns instead.

**Run every cell in order** with `Shift + Enter`.

In [ ]:
# --- setup: run me first! ---
import pandas as pd, numpy as np, matplotlib.pyplot as plt

DATA = "https://raw.githubusercontent.com/emcramer/psi-class-2026/main/data/processed"

plt.rcParams.update({"figure.figsize": (8, 5.5), "font.size": 13,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e1e0d9",
                     "figure.facecolor": "white", "axes.facecolor": "white"})
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"
print("ready")

In [ ]:
cells = pd.read_csv(f"{DATA}/cells_sample.csv")
print(cells.shape)
cells.head()

Each **row** is one cell. Each **column** is how much of one protein that cell
has. Think of it as a fingerprint, 16 numbers long.

The `celltype` column is the answer key — what biologists said each cell is.
**We're going to hide it**, let the computer group the cells on its own, and
then see whether it rediscovered the same answer.

---
## Part 1 — Clustering: finding groups nobody labeled

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

MARKERS = ["CD45", "CD3", "CD8", "CD4", "CD20", "CD68", "CD11c", "MPO",
           "Pan-Keratin", "Beta catenin", "Keratin17", "Vimentin", "SMA",
           "CD31", "FoxP3", "HLA-DR"]

X = StandardScaler().fit_transform(cells[MARKERS])   # put every protein on the same scale
labels = KMeans(n_clusters=6, n_init=20, random_state=0).fit_predict(X)

print("cells in each cluster:", np.bincount(labels))

The computer just sorted 20,000 cells into 6 piles — **without ever seeing the
answer key.** Now: what *is* each pile?

Let's look at which proteins are high in each cluster.

In [ ]:
profile = pd.DataFrame(X, columns=MARKERS).groupby(labels).mean()

plt.figure(figsize=(11, 4.5))
plt.imshow(profile, cmap="RdBu_r", vmin=-2.5, vmax=2.5, aspect="auto")
plt.xticks(range(len(MARKERS)), MARKERS, rotation=45, ha="right")
plt.yticks(range(6), [f"cluster {i}" for i in range(6)])
plt.colorbar(label="how much of this protein")
plt.grid(False); plt.title("Red = lots of it.  Blue = none of it.")
plt.show()

### 🔍 Your turn — be the biologist

Here's your cheat sheet:

| protein | means the cell is a... |
|---|---|
| **CD3, CD4, CD8** | T cell (the immune system's assassins) |
| **CD20** | B cell (makes antibodies) |
| **MPO** | neutrophil (first responder) |
| **CD68** | macrophage (the eater) |
| **Pan-Keratin, Keratin17** | tumor cell |
| **CD31** | blood vessel |
| **Vimentin, SMA** | structural/support cell |

Look at the red squares in each row and write down your guesses.
Then run the next cell to see how you did.

In [ ]:
answer = pd.crosstab(labels, cells["celltype"])
for i in range(6):
    top = profile.loc[i].sort_values(ascending=False).head(3)
    print(f"cluster {i}: high in {', '.join(top.index)}")
    print(f"           -> really mostly {answer.loc[i].idxmax()}\n")

The computer found T cells, B cells, neutrophils and tumor cells **on its
own**. That's *clustering*: finding groups when nobody gave you labels.

---
## Part 2 — Squashing 16 numbers into a picture

Every cell has 16 measurements, so each cell is a point in *16-dimensional*
space. We can't see that. **PCA** squashes it down to 2 so we can.

In [ ]:
from sklearn.decomposition import PCA

xy2 = PCA(n_components=2).fit_transform(X)

kind = np.where(cells.celltype == "Tumor", "tumor",
        np.where(cells.celltype.isin(["Endothelial", "Mesenchymal"]), "other",
                 "immune"))

plt.figure(figsize=(7.5, 6.5))
for k, c in [("other", GREEN), ("tumor", BLUE), ("immune", ORANGE)]:
    m = kind == k
    plt.scatter(xy2[m, 0], xy2[m, 1], s=4, c=c, alpha=0.5, label=k)
plt.legend(markerscale=4); plt.xticks([]); plt.yticks([]); plt.grid(False)
plt.title("Every dot is one cell")
plt.show()

Tumor cells land in one region, immune cells in another — and again, **the
colors were never used to make the picture.** The structure was already there.

---
## Part 3 — Does any of this predict who survives?

Now let's go from cells to *patients*. First question: does it matter **which
cells** a patient has?

In [ ]:
patients = pd.read_csv(f"{DATA}/patients.csv")
all_cells = pd.read_csv(f"{DATA}/cells_xy.csv.gz")

# what fraction of each patient's tumor is each cell type?
composition = pd.crosstab(all_cells.SampleID, all_cells.celltype, normalize="index")
composition = composition.loc[composition.index.isin(patients.SampleID)]

groups = KMeans(2, n_init=50, random_state=0).fit_predict(
    StandardScaler().fit_transform(composition))

df = pd.DataFrame({"SampleID": composition.index, "group": groups}).merge(patients)
print(df.groupby("group").size())

In [ ]:
def survival_curve(days, died):
    """Kaplan-Meier: the fraction still alive as time goes on."""
    order = np.argsort(days)
    days, died = np.array(days)[order], np.array(died)[order]
    t, s, alive, n = [0], [1.0], 1.0, len(days)
    for i, d in enumerate(days):
        if died[i]:
            alive *= 1 - 1 / n
            t.append(d / 365.25); s.append(alive)
        n -= 1
    return t + [days.max() / 365.25], s + [alive]

for g, c in [(0, BLUE), (1, ORANGE)]:
    s = df[df.group == g]
    t, surv = survival_curve(s.survival_days, s.event)
    plt.step(t, surv, where="post", color=c, lw=2.5, label=f"group {g} (n={len(s)})")
plt.ylim(0, 1.02); plt.xlabel("years"); plt.ylabel("fraction still alive")
plt.title("Grouped by WHICH cells they have"); plt.legend()
plt.show()

### 🤔 The lines sit on top of each other.

We grouped patients by which cells they have, and it tells us **nothing** about
who survives. That's a real result, and it's not a failure — it's a clue.

If *who* is in the tumor doesn't matter... maybe ***where*** they are does.

---
## Part 4 — Where the cells are

Here are two patients. Both are about **half immune cells**. Look at them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
for ax, pid in zip(axes, [3, 12]):
    g = all_cells[all_cells.SampleID == pid]
    g = g.assign(kind=np.where(g.celltype == "Tumor", "tumor",
                 np.where(g.celltype.isin(["Endothelial", "Mesenchymal"]),
                          "other", "immune")))
    for k, c in [("other", GREEN), ("tumor", BLUE), ("immune", ORANGE)]:
        s = g[g.kind == k]
        ax.scatter(s.x, s.y, s=3, c=c, alpha=0.75)
    pct = (g.kind == "immune").mean()
    ax.set_title(f"Patient {pid} — {pct:.0%} immune cells")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False); ax.set_aspect("equal")
plt.show()

Same ingredients, completely different architecture. On the left the immune
cells are **walled off** in their own territory. On the right they're **mixed**
right in among the tumor.

Let's turn that into a number. For each patient:

> **mixing score = (immune cells touching tumor cells) ÷ (immune cells touching each other)**

Low score = walled off. High score = mixed together.

In [ ]:
from scipy.spatial import cKDTree

all_cells["kind"] = np.where(all_cells.celltype == "Tumor", "tumor",
    np.where(all_cells.celltype.isin(["Endothelial", "Mesenchymal"]), "other", "immune"))

rows = []
for pid, g in all_cells.groupby("SampleID"):
    # every pair of cells within 30 pixels of each other = "touching"
    pairs = cKDTree(g[["x", "y"]].values).query_pairs(30, output_type="ndarray")
    a, b = g.kind.values[pairs[:, 0]], g.kind.values[pairs[:, 1]]

    immune_immune = ((a == "immune") & (b == "immune")).sum()
    immune_tumor = (((a == "immune") & (b == "tumor")) |
                     ((a == "tumor") & (b == "immune"))).sum()

    rows.append({"SampleID": pid,
                 "mixing_score": immune_tumor / max(immune_immune, 1),
                 "n_immune": (g.kind == "immune").sum()})

scores = pd.DataFrame(rows)
scores.head()

Two things before we use these scores:

1. Some tumors have almost **no** immune cells at all ("cold"). The ratio is
   meaningless for them, so we set them aside.
2. Everyone else we split at a cutoff of **0.26**.

In [ ]:
warm = scores[scores.n_immune >= 250].copy()
warm["our_label"] = np.where(warm.mixing_score < 0.26, "walled off", "mixed")

check = warm.merge(patients, on="SampleID")
match = (check.our_label.map({"walled off": "compartmentalized",
                              "mixed": "mixed"}) == check.published_class)
print(f"We agree with the published paper on {match.sum()} of {len(check)} patients.")

### 🎉 Every single one.

Those published labels took a research team at Stanford a lot of work. You just
reproduced them with about ten lines of code.

Now the real question — **does it predict survival?**

In [ ]:
for lab, c in [("walled off", BLUE), ("mixed", ORANGE)]:
    s = check[check.our_label == lab]
    t, surv = survival_curve(s.survival_days, s.event)
    plt.step(t, surv, where="post", color=c, lw=2.5, label=f"{lab} (n={len(s)})")
plt.ylim(0, 1.02); plt.xlabel("years"); plt.ylabel("fraction still alive")
plt.title("Grouped by WHERE their cells are"); plt.legend()
plt.show()

## That gap is the whole point.

Patients whose immune cells were **mixed in** with the tumor did far worse —
about **5× the risk of dying** — than patients whose immune cells were **walled
off**. Same disease, same cell types, different architecture.

*Which* cells you have: told us nothing.
*Where* they are: told us a lot.

---
### 🚀 If you have time

- Change `30` to `50` in the mixing-score cell. Do the results survive?
- Change the cutoff `0.26`. How much can you move it before the answer changes?
- Try `n_clusters=3` or `10` in Part 1. What new cell types appear?